# 02 - Qualidade e pré-processamento dos dados

## 1. Objetivo do notebook

Este notebook avalia a qualidade da base horária consolidada da estação A701, identificando ausências, códigos especiais, duplicidades e interrupções temporais. Também transforma os registros horários do INMET, originalmente em UTC, em acumulados e médias por **dia civil de São Paulo** (`America/Sao_Paulo`).

A correção considera as mudanças históricas de horário de verão (dias de 23, 24 ou 25 horas) e mantém a distinção entre precipitação nula e precipitação indisponível.

**Nota de reprodutibilidade:** esta versão revisada deve ser executada desde o início. A base Gold anterior não é sobrescrita: o resultado será exportado para um arquivo específico com sufixo `horario_sp`. Os notebooks posteriores deverão ser atualizados e reexecutados após a validação desta base.

**Convenção temporal:** a conversão associa cada medição horária ao timestamp UTC indicado na origem. A interpretação do intervalo físico representado pelo acumulado horário do INMET merece confirmação documental antes de eventual uso operacional. Não se aplica deslocamento manual adicional de uma hora.

## 2. Bibliotecas e carregamento da base Silver

Nesta etapa, são importadas as bibliotecas utilizadas no pré-processamento e carregada a base horária consolidada produzida no notebook anterior, armazenada na camada Silver.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


In [2]:
pasta_silver = Path("../data/silver")

arquivo_silver = pasta_silver / "a701_2010_2025_horario_consolidado.parquet"

print(arquivo_silver)

..\data\silver\a701_2010_2025_horario_consolidado.parquet


In [3]:
df = pd.read_parquet(arquivo_silver)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

Linhas: 140,256
Colunas: 20


## 3. Validação estrutural inicial

Nesta etapa, é verificada a estrutura da base consolidada, incluindo tipos das variáveis, intervalo temporal e quantidade de registros.


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 140256 entries, 0 to 140255
Data columns (total 20 columns):
 #   Column                                                 Non-Null Count   Dtype         
---  ------                                                 --------------   -----         
 0   DATA (YYYY-MM-DD)                                      140256 non-null  datetime64[us]
 1   HORA (UTC)                                             140256 non-null  str           
 2   PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                       139540 non-null  float64       
 3   PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  139976 non-null  float64       
 4   PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)        139972 non-null  float64       
 5   PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)       139972 non-null  float64       
 6   RADIACAO GLOBAL (KJ/m²)                                111257 non-null  float64       
 7   TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)           139976 non-n

In [5]:
print("Data mínima:", df["DATA (YYYY-MM-DD)"].min())
print("Data máxima:", df["DATA (YYYY-MM-DD)"].max())
print("Quantidade de registros:", len(df))
print("Quantidade de colunas:", df.shape[1])

Data mínima: 2010-01-01 00:00:00
Data máxima: 2025-12-31 00:00:00
Quantidade de registros: 140256
Quantidade de colunas: 20


## 4. Análise de valores ausentes e códigos especiais

Nesta etapa, são quantificados os valores ausentes e os códigos especiais presentes nas variáveis meteorológicas. O objetivo é distinguir ausência de informação de valores válidos, especialmente no caso da precipitação, em que o valor zero representa ausência de chuva e não deve ser tratado como dado faltante.


In [6]:
resumo_nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "percentual_nulos": (df.isna().mean() * 100).round(2)
})

resumo_nulos = resumo_nulos.sort_values(
    "percentual_nulos",
    ascending=False
)

resumo_nulos

,nulos,percentual_nulos
RADIACAO GLOBAL (KJ/m²),28999,20.68
"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",716,0.51
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),288,0.21
"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",280,0.20
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),286,0.20
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),284,0.20
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),284,0.20
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),284,0.20
TEMPERATURA DO PONTO DE ORVALHO (°C),283,0.20
TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),284,0.20


In [7]:
colunas_numericas = df.select_dtypes(include="number").columns

contagem_9999 = (df[colunas_numericas] == -9999).sum()

contagem_9999 = contagem_9999[
    contagem_9999 > 0
].sort_values(ascending=False)

contagem_9999

RADIACAO GLOBAL (KJ/m²)                                  36957
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)               613
TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)                 517
VENTO, RAJADA MAXIMA (m/s)                                 467
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                   466
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)           466
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                   466
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)           466
TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)                 466
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)           464
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)            464
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                           463
TEMPERATURA DO PONTO DE ORVALHO (°C)                       461
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                       461
UMIDADE RELATIVA DO AR, HORARIA (%)                        461
VENTO, VELOCIDADE HORARIA (m/s)                        

### Interpretação da qualidade inicial

A análise mostrou que a base possui dois tipos de ausência de informação: valores nulos (`NaN`) e registros codificados como `-9999`. Como esses dois casos representam indisponibilidade de informação, ambos foram considerados na avaliação de completude.

A precipitação apresentou 1.179 registros horários indisponíveis ao considerar conjuntamente os valores nulos e os códigos `-9999`. As variáveis auxiliares apresentaram baixa proporção de ausência, com exceção da radiação global, que mostrou um padrão de indisponibilidade fortemente relacionado ao horário do dia.

In [8]:
df["ano"] = df["DATA (YYYY-MM-DD)"].dt.year

In [9]:
variaveis_principais = [
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "VENTO, VELOCIDADE HORARIA (m/s)",
    "RADIACAO GLOBAL (KJ/m²)"
]

In [10]:
qualidade_por_ano = []

for ano, grupo in df.groupby("ano"):
    for coluna in variaveis_principais:
        nulos = grupo[coluna].isna().sum()
        cod_9999 = (grupo[coluna] == -9999).sum()

        total_indisponivel = nulos + cod_9999
        percentual = total_indisponivel / len(grupo) * 100

        qualidade_por_ano.append({
            "ano": ano,
            "variavel": coluna,
            "nulos": nulos,
            "codigo_-9999": cod_9999,
            "total_indisponivel": total_indisponivel,
            "percentual_indisponivel": round(percentual, 2)
        })

qualidade_por_ano = pd.DataFrame(qualidade_por_ano)

qualidade_por_ano.head()

,ano,variavel,nulos,codigo_-9999,total_indisponivel,percentual_indisponivel
0,2010,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",0,10,10,0.11
1,2010,"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",0,162,162,1.85
2,2010,"UMIDADE RELATIVA DO AR, HORARIA (%)",0,10,10,0.11
3,2010,"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORAR...",0,10,10,0.11
4,2010,"VENTO, VELOCIDADE HORARIA (m/s)",0,10,10,0.11


In [11]:
tabela_qualidade_anual = qualidade_por_ano.pivot(
    index="ano",
    columns="variavel",
    values="percentual_indisponivel"
)

tabela_qualidade_anual

variavel,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",RADIACAO GLOBAL (KJ/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)","UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, VELOCIDADE HORARIA (m/s)"
ano,,,,,,
2010,0.11,0.11,46.19,1.85,0.11,0.11
2011,0.26,0.26,46.56,0.26,0.26,0.26
2012,0.43,0.43,46.35,0.43,0.43,0.43
2013,0.53,0.53,46.15,0.53,0.53,0.53
2014,2.33,2.33,47.72,2.33,2.33,2.33
2015,0.00,0.00,46.27,0.00,0.00,0.00
2016,0.75,0.75,47.68,0.76,0.76,0.75
2017,0.83,0.83,47.64,0.83,0.83,0.84
2018,0.03,0.00,47.08,0.00,0.00,0.00


## 5. Padronização temporal e distribuição das falhas

Os arquivos do INMET trazem datas e horas em UTC, com formatos de hora diferentes entre os anos. Primeiro padronizamos o formato e construímos um timestamp UTC válido; em seguida, convertemos cada instante para `America/Sao_Paulo`. A coluna `data_sp` será utilizada nas agregações diárias.

As análises de falhas por ano realizadas nas seções iniciais são diagnósticos da data original UTC. A análise mensal abaixo, após a conversão, utiliza calendário local.

In [12]:
df["datetime"] = pd.to_datetime(
    df["DATA (YYYY-MM-DD)"].astype(str)
    + " "
    + df["HORA (UTC)"].astype(str),
    errors="coerce"
)

print("Datetimes inválidos:", df["datetime"].isna().sum())

Datetimes inválidos: 61368


### Diagnóstico do formato de hora

A tentativa inicial de combinar diretamente os campos de data e hora resultou em 61.368 timestamps inválidos. A inspeção mostrou que o formato do campo de hora mudou a partir de 2019, passando de `HH:MM` para `HHMM UTC`.

Por esse motivo, o campo de hora foi padronizado antes da criação definitiva do timestamp. Esse diagnóstico foi mantido no notebook para documentar a inconsistência encontrada na fonte e a decisão adotada para corrigi-la.


In [13]:
for ano, grupo in df.groupby("ano"):
    primeira_hora = grupo["HORA (UTC)"].iloc[0]

    print(
        f"{ano} -> {primeira_hora}"
    )

2010 -> 00:00
2011 -> 00:00
2012 -> 00:00
2013 -> 00:00
2014 -> 00:00
2015 -> 00:00
2016 -> 00:00
2017 -> 00:00
2018 -> 00:00
2019 -> 0000 UTC
2020 -> 0000 UTC
2021 -> 0000 UTC
2022 -> 0000 UTC
2023 -> 0000 UTC
2024 -> 0000 UTC
2025 -> 0000 UTC


In [14]:
def padronizar_hora(valor):
    valor = str(valor).strip()

    if "UTC" in valor:
        valor = valor.replace(" UTC", "")
        return f"{valor[:2]}:{valor[2:]}"

    return valor

In [15]:
df["hora_padronizada"] = df["HORA (UTC)"].apply(padronizar_hora)

In [16]:
for ano, grupo in df.groupby("ano"):
    print(
        ano,
        "-> original:",
        grupo["HORA (UTC)"].iloc[0],
        "| padronizada:",
        grupo["hora_padronizada"].iloc[0]
    )

2010 -> original: 00:00 | padronizada: 00:00
2011 -> original: 00:00 | padronizada: 00:00
2012 -> original: 00:00 | padronizada: 00:00
2013 -> original: 00:00 | padronizada: 00:00
2014 -> original: 00:00 | padronizada: 00:00
2015 -> original: 00:00 | padronizada: 00:00
2016 -> original: 00:00 | padronizada: 00:00
2017 -> original: 00:00 | padronizada: 00:00
2018 -> original: 00:00 | padronizada: 00:00
2019 -> original: 0000 UTC | padronizada: 00:00
2020 -> original: 0000 UTC | padronizada: 00:00
2021 -> original: 0000 UTC | padronizada: 00:00
2022 -> original: 0000 UTC | padronizada: 00:00
2023 -> original: 0000 UTC | padronizada: 00:00
2024 -> original: 0000 UTC | padronizada: 00:00
2025 -> original: 0000 UTC | padronizada: 00:00


In [17]:
# Combina os campos de data/hora da fonte, que estão em UTC.
df["datetime"] = pd.to_datetime(
    df["DATA (YYYY-MM-DD)"].dt.strftime("%Y-%m-%d")
    + " "
    + df["hora_padronizada"],
    format="%Y-%m-%d %H:%M",
    errors="coerce",
    utc=True
)

if df["datetime"].isna().any():
    raise ValueError("Há timestamps UTC inválidos; corrigir antes de agregar.")

# São Paulo teve horário de verão em parte da série 2010–2019.
df["datetime_sp"] = df["datetime"].dt.tz_convert("America/Sao_Paulo")
df["data_sp"] = df["datetime_sp"].dt.tz_localize(None).dt.normalize()

print("Datetimes inválidos:", df["datetime"].isna().sum())
print("Datetime UTC mínimo:", df["datetime"].min())
print("Datetime UTC máximo:", df["datetime"].max())
print("Primeira data local:", df["data_sp"].min())
print("Última data local:", df["data_sp"].max())
display(df[["datetime", "datetime_sp", "data_sp"]].head())

Datetimes inválidos: 0
Datetime UTC mínimo: 2010-01-01 00:00:00+00:00
Datetime UTC máximo: 2025-12-31 23:00:00+00:00
Primeira data local: 2009-12-31 00:00:00
Última data local: 2025-12-31 00:00:00


,datetime,datetime_sp,data_sp
0,2010-01-01 00:00:00+00:00,2009-12-31 22:00:00-02:00,2009-12-31
1,2010-01-01 01:00:00+00:00,2009-12-31 23:00:00-02:00,2009-12-31
2,2010-01-01 02:00:00+00:00,2010-01-01 00:00:00-02:00,2010-01-01
3,2010-01-01 03:00:00+00:00,2010-01-01 01:00:00-02:00,2010-01-01
4,2010-01-01 04:00:00+00:00,2010-01-01 02:00:00-02:00,2010-01-01


### Calendário civil de São Paulo e duração real dos dias

A mudança histórica do horário de verão pode produzir dias de 23 e 25 horas e até meias-noites inexistentes. Por isso, o calendário de referência é calculado primeiro por uma sequência UTC contínua e depois convertido para São Paulo. Não se tenta gerar uma sequência de meias-noites locais diretamente.

In [18]:
# Número esperado de horas de cada data local: 23, 24 ou 25.
# Criamos horas UTC adicionais em ambas as extremidades, para que os dias
# parcialmente cobertos pela coleta sejam reconhecidos como incompletos.
calendario_utc = pd.date_range(
    start=df["datetime"].min() - pd.Timedelta(days=2),
    end=df["datetime"].max() + pd.Timedelta(days=2),
    freq="h",
    tz="UTC"
)

calendario_sp = calendario_utc.tz_convert("America/Sao_Paulo")
horas_esperadas_sp = (
    pd.DataFrame({
        "data_sp": pd.to_datetime(calendario_sp.tz_localize(None).normalize())
    })
    .groupby("data_sp", as_index=False)
    .size()
    .rename(columns={"size": "horas_esperadas"})
)

# Apenas dias integralmente cobertos pelo calendário auxiliar.
horas_esperadas_sp = horas_esperadas_sp[
    horas_esperadas_sp["horas_esperadas"].between(23, 25)
].copy()

print("Duração dos dias civis locais (23/24/25 h):")
display(horas_esperadas_sp["horas_esperadas"].value_counts().sort_index())

Duração dos dias civis locais (23/24/25 h):


horas_esperadas
23       9
24    5828
25      10
Name: count, dtype: int64

In [19]:
col_precipitacao = "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)"

df["precipitacao_indisponivel"] = (
    df[col_precipitacao].isna()
    | (df[col_precipitacao] == -9999)
)

In [20]:
falhas_precipitacao_ano = (
    df.groupby("ano")["precipitacao_indisponivel"]
      .agg(["sum", "mean"])
)

falhas_precipitacao_ano["percentual"] = (
    falhas_precipitacao_ano["mean"] * 100
).round(2)

falhas_precipitacao_ano[["sum", "percentual"]]

,sum,percentual
ano,,
2010,10,0.11
2011,23,0.26
2012,38,0.43
2013,46,0.53
2014,204,2.33
2015,0,0.00
2016,66,0.75
2017,73,0.83
2018,3,0.03


In [21]:
# Contagem de falhas por mês civil local (não por mês UTC).
falhas_precipitacao_mes = (
    df.loc[df["precipitacao_indisponivel"]]
    .groupby([
        df["data_sp"].dt.year.rename("ano"),
        df["data_sp"].dt.month.rename("mes")
    ])
    .size()
    .unstack(fill_value=0)
)

falhas_precipitacao_mes

mes,1,2,3,4,5,6,7,8,9,10,11,12
ano,,,,,,,,,,,,
2010,2,0,0,0,0,0,2,3,2,0,1,0
2011,0,0,10,0,3,4,3,2,0,0,1,0
2012,0,0,25,0,1,1,5,3,1,2,0,0
2013,0,0,18,1,0,2,12,7,3,1,2,0
2014,0,1,0,0,0,0,203,0,0,0,0,0
2016,0,66,0,0,0,0,0,0,0,0,0,0
2017,0,0,0,0,73,0,0,0,0,0,0,0
2018,0,0,0,0,1,0,0,2,0,0,0,0
2020,0,0,0,0,0,0,0,1,0,1,0,0


In [22]:
falhas = df.loc[
    df["precipitacao_indisponivel"],
    ["datetime"]
].copy()

falhas["diferenca_horas"] = falhas["datetime"].diff().dt.total_seconds().div(3600)

falhas["novo_bloco"] = falhas["diferenca_horas"].ne(1)

falhas["bloco"] = falhas["novo_bloco"].cumsum()

blocos_falha = (
    falhas.groupby("bloco")
    .agg(
        inicio=("datetime", "min"),
        fim=("datetime", "max"),
        horas_faltantes=("datetime", "size")
    )
    .sort_values("horas_faltantes", ascending=False)
)

blocos_falha.head(20)

,inicio,fim,horas_faltantes
bloco,,,
109,2023-09-03 13:00:00+00:00,2023-09-12 15:00:00+00:00,219
108,2023-08-24 21:00:00+00:00,2023-09-02 20:00:00+00:00,216
67,2014-07-20 06:00:00+00:00,2014-07-28 16:00:00+00:00,203
68,2016-02-08 21:00:00+00:00,2016-02-11 14:00:00+00:00,66
119,2023-11-26 07:00:00+00:00,2023-11-28 12:00:00+00:00,54
70,2017-05-20 22:00:00+00:00,2017-05-22 16:00:00+00:00,43
69,2017-05-19 07:00:00+00:00,2017-05-20 12:00:00+00:00,30
118,2023-11-21 10:00:00+00:00,2023-11-22 13:00:00+00:00,28
25,2012-03-07 13:00:00+00:00,2012-03-08 12:00:00+00:00,24


### Interpretação das falhas de precipitação

Foram identificados blocos consecutivos de ausência com duração elevada, incluindo períodos superiores a 200 horas em 2014 e 2023.

Esse padrão indica que parte das falhas corresponde a interrupções prolongadas de medição, e não apenas a ausências pontuais. Por esse motivo, optou-se por não realizar imputação da precipitação horária nesses períodos, evitando a criação artificial de valores para a variável-alvo.

## 6. Completude e tratamento da precipitação

A soma diária será calculada pelo **calendário civil de São Paulo**. Um dia só será válido quando todas as horas esperadas para aquela data (23, 24 ou 25, conforme o histórico do horário de verão) estiverem registradas e apresentarem precipitação disponível. Dias incompletos não são imputados nem tratados como dias secos.

As bordas da coleta merecem atenção: alguns timestamps UTC de 01/01/2010 pertencem ao dia local 31/12/2009, e o dia local 31/12/2025 fica parcialmente coberto. Ambas as situações são identificadas pela regra de completude.

In [23]:
# Qualidade da precipitação por dia local, com horas esperadas do calendário.
qualidade_diaria_precipitacao = (
    df.assign(
        precipitacao_valida=(
            df[col_precipitacao].notna()
            & (df[col_precipitacao] != -9999)
        )
    )
    .groupby("data_sp", as_index=False)
    .agg(
        horas_registradas=("datetime", "size"),
        horas_validas_precipitacao=("precipitacao_valida", "sum")
    )
    .merge(horas_esperadas_sp, on="data_sp", how="left", validate="one_to_one")
)

if qualidade_diaria_precipitacao["horas_esperadas"].isna().any():
    raise ValueError("O calendário não cobre todas as datas locais.")

qualidade_diaria_precipitacao["horas_indisponiveis"] = (
    qualidade_diaria_precipitacao["horas_esperadas"]
    - qualidade_diaria_precipitacao["horas_validas_precipitacao"]
)

qualidade_diaria_precipitacao["dia_completo"] = (
    qualidade_diaria_precipitacao["horas_registradas"].eq(
        qualidade_diaria_precipitacao["horas_esperadas"]
    )
    & qualidade_diaria_precipitacao["horas_validas_precipitacao"].eq(
        qualidade_diaria_precipitacao["horas_esperadas"]
    )
)

display(qualidade_diaria_precipitacao.head())

,data_sp,horas_registradas,horas_validas_precipitacao,horas_esperadas,horas_indisponiveis,dia_completo
0,2009-12-31,2,2,24,22,False
1,2010-01-01,24,24,24,0,True
2,2010-01-02,24,24,24,0,True
3,2010-01-03,24,24,24,0,True
4,2010-01-04,24,24,24,0,True


In [24]:
qualidade_diaria_precipitacao["horas_indisponiveis"].value_counts().sort_index()

horas_indisponiveis
0     5652
1      106
2       17
3        7
4        4
5        3
6        2
7        2
8        2
9        3
10       2
11       3
12       1
13       2
14       4
15       2
17       1
18       1
20       2
21       1
22       1
24      27
Name: count, dtype: int64

In [25]:
# Mostra a auditoria completa (inclui o dia parcial 31/12/2009)
# e a série efetiva do TCC (01/01/2010 a 31/12/2025).
periodo_tcc = qualidade_diaria_precipitacao["data_sp"].between(
    "2010-01-01", "2025-12-31"
)

audit_tcc = qualidade_diaria_precipitacao.loc[periodo_tcc]
total_dias = len(audit_tcc)
dias_completos = audit_tcc["dia_completo"].sum()
dias_incompletos = total_dias - dias_completos

print("Datas locais auditadas (incluindo borda de 2009):", len(qualidade_diaria_precipitacao))
print(f"Dias do TCC: {total_dias:,}")
print(f"Dias completos de precipitação: {dias_completos:,}")
print(f"Dias incompletos de precipitação: {dias_incompletos:,}")
print(f"Disponibilidade: {dias_completos / total_dias * 100:.2f}%")

Datas locais auditadas (incluindo borda de 2009): 5845
Dias do TCC: 5,844
Dias completos de precipitação: 5,652
Dias incompletos de precipitação: 192
Disponibilidade: 96.71%


In [26]:
distribuicao_horas_faltantes = (
    qualidade_diaria_precipitacao.loc[
        qualidade_diaria_precipitacao["horas_indisponiveis"] > 0,
        "horas_indisponiveis"
    ]
    .value_counts()
    .sort_index()
)

distribuicao_horas_faltantes

horas_indisponiveis
1     106
2      17
3       7
4       4
5       3
6       2
7       2
8       2
9       3
10      2
11      3
12      1
13      2
14      4
15      2
17      1
18      1
20      2
21      1
22      1
24     27
Name: count, dtype: int64

In [27]:
def classificar_falha(horas):
    if horas == 0:
        return "Completo"
    elif horas <= 2:
        return "1-2 horas"
    elif horas <= 6:
        return "3-6 horas"
    elif horas <= 12:
        return "7-12 horas"
    elif horas <= 23:
        return "13-23 horas"
    else:
        return "24 ou mais horas"

qualidade_diaria_precipitacao["faixa_falha"] = (
    qualidade_diaria_precipitacao["horas_indisponiveis"]
    .apply(classificar_falha)
)

qualidade_diaria_precipitacao["faixa_falha"].value_counts()

faixa_falha
Completo            5652
1-2 horas            123
24 ou mais horas      27
3-6 horas             16
13-23 horas           14
7-12 horas            13
Name: count, dtype: int64

### Critério de completude da precipitação diária

A precipitação horária é somada somente quando **todas** as observações esperadas do dia local estão disponíveis. A duração do dia varia entre 23, 24 e 25 horas em razão do horário de verão histórico de São Paulo. A completude compara tanto o número de linhas quanto o número de medições válidas com a duração esperada.

A auditoria inicial da série convertida identificou **5.652 dias válidos e 192 incompletos** no período de 2010 a 2025 (a data parcial de 31/12/2009 é excluída). Esses números devem ser conferidos ao reexecutar o notebook. Dias com informação incompleta permanecem como `NaN`.

In [28]:
df["precipitacao_horaria"] = (
    df[col_precipitacao]
    .replace(-9999, np.nan)
)

In [29]:
print(
    "Valores -9999 restantes:",
    (df["precipitacao_horaria"] == -9999).sum()
)

print(
    "Valores ausentes:",
    df["precipitacao_horaria"].isna().sum()
)

Valores -9999 restantes: 0
Valores ausentes: 1179


In [30]:
# Soma precipitação por DATA LOCAL; somente dias integralmente válidos.
precipitacao_diaria = (
    df.groupby("data_sp", as_index=False)
    .agg(
        horas_validas=("precipitacao_horaria", "count"),
        precipitacao_diaria_mm=(
            "precipitacao_horaria", lambda x: x.sum(min_count=1)
        )
    )
    .merge(
        qualidade_diaria_precipitacao[
            ["data_sp", "horas_registradas", "horas_esperadas", "dia_completo"]
        ],
        on="data_sp", how="left", validate="one_to_one"
    )
)

precipitacao_diaria.loc[
    ~precipitacao_diaria["dia_completo"], "precipitacao_diaria_mm"
] = np.nan

precipitacao_diaria = precipitacao_diaria.rename(
    columns={"data_sp": "DATA (YYYY-MM-DD)"}
)

# Limita o calendário ao período formal do TCC.
precipitacao_diaria = precipitacao_diaria.loc[
    precipitacao_diaria["DATA (YYYY-MM-DD)"].between(
        "2010-01-01", "2025-12-31"
    )
].copy()

precipitacao_diaria.head()

,DATA (YYYY-MM-DD),horas_validas,precipitacao_diaria_mm,horas_registradas,horas_esperadas,dia_completo
1,2010-01-01,24,6.4,24,24,True
2,2010-01-02,24,36.4,24,24,True
3,2010-01-03,24,0.4,24,24,True
4,2010-01-04,24,13.4,24,24,True
5,2010-01-05,24,50.2,24,24,True


In [31]:
# Validação direta do resultado no período 2010–2025.
print("Datas:", len(precipitacao_diaria))
print("Dias com precipitação válida:", precipitacao_diaria["precipitacao_diaria_mm"].notna().sum())
print("Dias sem acumulado válido:", precipitacao_diaria["precipitacao_diaria_mm"].isna().sum())
print("Duração do dia local:")
display(precipitacao_diaria["horas_esperadas"].value_counts().sort_index())

assert len(precipitacao_diaria) == 5844, "Quantidade inesperada de dias na série."
assert precipitacao_diaria["dia_completo"].equals(
    precipitacao_diaria["precipitacao_diaria_mm"].notna()
), "Divergência entre completude e precipitação calculada."

Datas: 5844
Dias com precipitação válida: 5652
Dias sem acumulado válido: 192
Duração do dia local:


horas_esperadas
23       9
24    5825
25      10
Name: count, dtype: int64

## 7. Avaliação das variáveis auxiliares

Após o tratamento da precipitação, são avaliadas as principais variáveis meteorológicas auxiliares: temperatura, umidade relativa, pressão atmosférica e velocidade do vento. A radiação global é analisada separadamente devido ao seu padrão específico de disponibilidade ao longo do dia.


In [32]:
variaveis_auxiliares = {
    "temperatura_horaria": "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "umidade_horaria": "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "pressao_horaria": "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "vento_horaria": "VENTO, VELOCIDADE HORARIA (m/s)",
    "radiacao_horaria": "RADIACAO GLOBAL (KJ/m²)"
}

for nova_coluna, coluna_original in variaveis_auxiliares.items():
    df[nova_coluna] = df[coluna_original].replace(-9999, np.nan)

In [33]:
resumo_auxiliares = pd.DataFrame({
    "nulos": df[list(variaveis_auxiliares.keys())].isna().sum(),
    "percentual_nulos": (
        df[list(variaveis_auxiliares.keys())].isna().mean() * 100
    ).round(2)
})

resumo_auxiliares

,nulos,percentual_nulos
temperatura_horaria,893,0.64
umidade_horaria,742,0.53
pressao_horaria,740,0.53
vento_horaria,743,0.53
radiacao_horaria,65956,47.03


In [34]:
# A análise da radiação por hora é feita em hora civil local.
df["hora"] = df["datetime_sp"].dt.hour

radiacao_por_hora = (
    df.groupby("hora")["radiacao_horaria"]
    .agg(total_registros="size", registros_validos="count")
)

radiacao_por_hora["indisponiveis"] = (
    radiacao_por_hora["total_registros"]
    - radiacao_por_hora["registros_validos"]
)

radiacao_por_hora["percentual_indisponivel"] = (
    radiacao_por_hora["indisponiveis"]
    / radiacao_por_hora["total_registros"] * 100
).round(2)

radiacao_por_hora

,total_registros,registros_validos,indisponiveis,percentual_indisponivel
hora,,,,
0,5835,82,5753,98.59
1,5844,83,5761,98.58
2,5844,67,5777,98.85
3,5844,84,5760,98.56
4,5844,84,5760,98.56
5,5844,95,5749,98.37
6,5844,1474,4370,74.78
7,5844,5716,128,2.19
8,5844,5799,45,0.77


In [35]:
radiacao_original = "RADIACAO GLOBAL (KJ/m²)"

radiacao_tipo_falha_hora = pd.DataFrame({
    "nulos": (
        df[radiacao_original]
        .isna()
        .groupby(df["hora"])
        .sum()
    ),
    "codigo_-9999": (
        (df[radiacao_original] == -9999)
        .groupby(df["hora"])
        .sum()
    )
})

radiacao_tipo_falha_hora["total"] = (
    radiacao_tipo_falha_hora["nulos"]
    + radiacao_tipo_falha_hora["codigo_-9999"]
)

radiacao_tipo_falha_hora

,nulos,codigo_-9999,total
hora,,,
0,2520,3233,5753
1,2528,3233,5761
2,2532,3245,5777
3,2519,3241,5760
4,2523,3237,5760
5,2512,3237,5749
6,1531,2839,4370
7,50,78,128
8,20,25,45


In [36]:
radiacao_valida_por_hora = (
    df.groupby("hora")["radiacao_horaria"]
    .agg(
        quantidade="count",
        media="mean",
        mediana="median",
        minimo="min",
        maximo="max"
    )
    .round(2)
)

radiacao_valida_por_hora

,quantidade,media,mediana,minimo,maximo
hora,,,,,
0,82,0.81,0.50,0.0,4.4
1,83,0.68,0.40,0.0,3.2
2,67,0.70,0.40,0.0,4.7
3,84,0.77,0.45,0.0,3.3
4,84,0.66,0.40,0.0,3.3
5,95,0.61,0.30,0.0,4.4
6,1474,33.69,17.70,0.0,280.1
7,5716,135.47,63.40,0.0,1157.7
8,5799,559.79,432.20,7.4,2239.3


### Avaliação da radiação global

A radiação global apresentou 47,03% de registros indisponíveis na série horária. Entretanto, a análise por horário evidenciou forte concentração das ausências durante o período noturno, enquanto as horas diurnas apresentaram elevada disponibilidade e valores coerentes com o ciclo diário de radiação solar. Devido a esse padrão estrutural de ausência e visando evitar imputações baseadas em pressupostos não observados diretamente nos dados, a variável não será considerada inicialmente entre os preditores meteorológicos principais, permanecendo disponível para análises experimentais posteriores.


## 8. Validação temporal e preparação da base diária

Conferimos a unicidade e a continuidade **em UTC** (o eixo que não possui saltos de horário de verão) e agregamos temperatura, umidade, pressão e vento pelo **dia local de São Paulo**. O critério de representatividade das médias auxiliares será proporcional à duração do dia, preservando a referência original de pelo menos 20/24, ou 83,3% de observações válidas.

Dias localmente parciais nas extremidades não serão considerados representativos para as médias auxiliares.

In [37]:
duplicados_datetime = df["datetime"].duplicated().sum()
print("Timestamps UTC duplicados:", duplicados_datetime)

sequencia_utc = pd.date_range(
    df["datetime"].min(), df["datetime"].max(), freq="h", tz="UTC"
)
quantidade_ausentes = len(
    sequencia_utc.difference(pd.DatetimeIndex(df["datetime"]))
)
print("Horas ausentes da sequência UTC:", quantidade_ausentes)

assert duplicados_datetime == 0, "Existem timestamps duplicados."
assert quantidade_ausentes == 0, "Existem lacunas na sequência UTC."

Timestamps UTC duplicados: 0
Horas ausentes da sequência UTC: 0


In [38]:
datas_ordenadas = df["datetime"].sort_values()

diferencas = datas_ordenadas.diff()

diferencas.value_counts().head(10)

datetime
0 days 01:00:00    140255
Name: count, dtype: int64

### Validação da continuidade temporal

A auditoria inicial da base de 2010–2025 confirmou **140.256 registros horários**, sem timestamps UTC duplicados ou horários ausentes na sequência. A continuidade deve ser avaliada em UTC, já que os horários civis de São Paulo incluem saltos e repetições durante as mudanças históricas do horário de verão.

A inexistência de lacunas nos timestamps não implica ausência de valores meteorológicos indisponíveis. Esses casos continuam representados como `NaN` após o tratamento dos códigos especiais.

In [39]:
variaveis_diarias = {
    "temperatura_horaria": "temperatura_media",
    "umidade_horaria": "umidade_media",
    "pressao_horaria": "pressao_media",
    "vento_horaria": "vento_medio"
}

agregacoes = {}
for coluna_horaria, coluna_diaria in variaveis_diarias.items():
    agregacoes[coluna_diaria] = (coluna_horaria, "mean")
    agregacoes[f"{coluna_diaria}_horas_validas"] = (coluna_horaria, "count")

auxiliares_diarias = (
    df.groupby("data_sp", as_index=False)
    .agg(**agregacoes)
    .merge(
        qualidade_diaria_precipitacao[
            ["data_sp", "horas_registradas", "horas_esperadas"]
        ],
        on="data_sp", how="left", validate="one_to_one"
    )
)

# Mínimo equivalente a 20/24 h (~83,3%) para dias de 23, 24 ou 25 h.
auxiliares_diarias["min_horas_auxiliares"] = np.ceil(
    auxiliares_diarias["horas_esperadas"] * (20 / 24)
).astype(int)

# Não aceitar dias locais cobertos parcialmente pela coleta.
auxiliares_diarias["cobertura_temporal_completa"] = (
    auxiliares_diarias["horas_registradas"]
    == auxiliares_diarias["horas_esperadas"]
)

auxiliares_diarias.head()

,data_sp,temperatura_media,temperatura_media_horas_validas,umidade_media,umidade_media_horas_validas,pressao_media,pressao_media_horas_validas,vento_medio,vento_medio_horas_validas,horas_registradas,horas_esperadas,min_horas_auxiliares,cobertura_temporal_completa
0,2009-12-31,21.650000,2,84.000000,2,924.100000,2,2.400000,2,2,24,20,False
1,2010-01-01,21.212500,24,87.333333,24,923.108333,24,2.141667,24,24,24,20,True
2,2010-01-02,22.279167,24,84.041667,24,925.154167,24,1.179167,24,24,24,20,True
3,2010-01-03,24.370833,24,77.666667,24,926.783333,24,1.550000,24,24,24,20,True
4,2010-01-04,24.200000,24,74.416667,24,926.600000,24,1.804167,24,24,24,20,True


In [40]:
colunas_horas_validas = [
    "temperatura_media_horas_validas",
    "umidade_media_horas_validas",
    "pressao_media_horas_validas",
    "vento_medio_horas_validas"
]

auxiliares_diarias[colunas_horas_validas].describe()

,temperatura_media_horas_validas,umidade_media_horas_validas,pressao_media_horas_validas,vento_medio_horas_validas
count,5845.000000,5845.000000,5845.000000,5845.000000
mean,23.843114,23.868948,23.869290,23.868777
std,1.572084,1.375423,1.375331,1.379444
min,0.000000,0.000000,0.000000,0.000000
25%,24.000000,24.000000,24.000000,24.000000
50%,24.000000,24.000000,24.000000,24.000000
75%,24.000000,24.000000,24.000000,24.000000
max,25.000000,25.000000,25.000000,25.000000


In [41]:
distribuicao_horas_validas = pd.DataFrame({
    coluna: auxiliares_diarias[coluna]
        .value_counts()
        .sort_index()
    for coluna in colunas_horas_validas
}).fillna(0).astype(int)

distribuicao_horas_validas

,temperatura_media_horas_validas,umidade_media_horas_validas,pressao_media_horas_validas,vento_medio_horas_validas
0,16,11,11,11
2,2,1,1,1
3,1,1,1,1
4,2,2,2,2
7,1,1,1,1
9,2,2,2,3
10,3,3,3,2
11,1,1,1,1
12,1,1,1,1
13,3,3,3,4


In [42]:
auxiliares_diarias_tratadas = auxiliares_diarias.copy()

In [43]:
regras_auxiliares = {
    "temperatura_media": "temperatura_media_horas_validas",
    "umidade_media": "umidade_media_horas_validas",
    "pressao_media": "pressao_media_horas_validas",
    "vento_medio": "vento_medio_horas_validas"
}

for variavel, coluna_horas in regras_auxiliares.items():
    auxiliares_diarias_tratadas.loc[
        (auxiliares_diarias_tratadas[coluna_horas]
         < auxiliares_diarias_tratadas["min_horas_auxiliares"])
        | ~auxiliares_diarias_tratadas["cobertura_temporal_completa"],
        variavel
    ] = np.nan

In [44]:
resumo_auxiliares_diarias = pd.DataFrame({
    "dias_validos": auxiliares_diarias_tratadas[
        list(regras_auxiliares.keys())
    ].notna().sum(),

    "dias_indisponiveis": auxiliares_diarias_tratadas[
        list(regras_auxiliares.keys())
    ].isna().sum()
})

resumo_auxiliares_diarias["percentual_disponivel"] = (
    resumo_auxiliares_diarias["dias_validos"]
    / len(auxiliares_diarias_tratadas)
    * 100
).round(2)

resumo_auxiliares_diarias

,dias_validos,dias_indisponiveis,percentual_disponivel
temperatura_media,5798,47,99.20
umidade_media,5805,40,99.32
pressao_media,5805,40,99.32
vento_medio,5805,40,99.32


### Critério de completude das variáveis auxiliares

Para temperatura, umidade relativa, pressão atmosférica e velocidade do vento, foram calculadas médias por dia **civil de São Paulo** com base nas observações horárias válidas. O limite mínimo foi mantido em aproximadamente 83,3% da duração local: `ceil(horas_esperadas × 20/24)`, resultando em **20 horas para dias de 23 ou 24 horas** e **21 horas para dias de 25 horas**. Exige-se também que a cobertura temporal do dia esteja completa, independentemente da disponibilidade de cada variável auxiliar.

Valores diários que não atendem aos requisitos permanecem ausentes. Não foi efetuada imputação nesta etapa.

In [45]:
variaveis_auditoria = [
    "precipitacao_horaria",
    "temperatura_horaria",
    "umidade_horaria",
    "pressao_horaria",
    "vento_horaria"
]

df[variaveis_auditoria].describe().T

,count,mean,std,min,25%,50%,75%,max
precipitacao_horaria,139077.0,0.179650,1.438464,0.0,0.0,0.0,0.0,82.0
temperatura_horaria,139363.0,20.538624,4.592903,4.1,17.3,20.3,23.4,37.3
umidade_horaria,139514.0,69.306464,17.835430,12.0,57.0,74.0,83.0,100.0
pressao_horaria,139516.0,927.155068,3.640624,913.1,924.6,927.0,929.6,940.3
vento_horaria,139513.0,1.845925,1.186329,0.0,1.0,1.8,2.6,10.9


In [46]:
print(
    "Precipitação negativa:",
    (df["precipitacao_horaria"] < 0).sum()
)

print(
    "Umidade fora de 0–100%:",
    (
        (df["umidade_horaria"] < 0)
        | (df["umidade_horaria"] > 100)
    ).sum()
)

print(
    "Vento negativo:",
    (df["vento_horaria"] < 0).sum()
)

Precipitação negativa: 0
Umidade fora de 0–100%: 0
Vento negativo: 0


### Verificação de valores inválidos

Não foram identificados valores negativos de precipitação ou velocidade do vento, nem registros de umidade relativa fora do intervalo de 0% a 100%.

Os valores mínimos e máximos observados para temperatura e pressão atmosférica também não apresentaram inconsistências evidentes. Dessa forma, não foram aplicados cortes adicionais baseados em limites arbitrários.

In [47]:
# Compatibiliza chaves diárias e elimina a data local parcial de 2009.
auxiliares_diarias_tratadas = auxiliares_diarias_tratadas.rename(
    columns={"data_sp": "DATA (YYYY-MM-DD)"}
)

# Não carrega as colunas de auditoria duas vezes no merge.
colunas_auxiliares_exportar = [
    "DATA (YYYY-MM-DD)",
    "temperatura_media", "umidade_media", "pressao_media", "vento_medio",
    *colunas_horas_validas
]

df_diario = precipitacao_diaria.merge(
    auxiliares_diarias_tratadas[colunas_auxiliares_exportar],
    on="DATA (YYYY-MM-DD)", how="left", validate="one_to_one"
)

df_diario = df_diario.sort_values("DATA (YYYY-MM-DD)").reset_index(drop=True)
display(df_diario.head())

,DATA (YYYY-MM-DD),horas_validas,precipitacao_diaria_mm,horas_registradas,horas_esperadas,dia_completo,temperatura_media,umidade_media,pressao_media,vento_medio,temperatura_media_horas_validas,umidade_media_horas_validas,pressao_media_horas_validas,vento_medio_horas_validas
0,2010-01-01,24,6.4,24,24,True,21.212500,87.333333,923.108333,2.141667,24,24,24,24
1,2010-01-02,24,36.4,24,24,True,22.279167,84.041667,925.154167,1.179167,24,24,24,24
2,2010-01-03,24,0.4,24,24,True,24.370833,77.666667,926.783333,1.550000,24,24,24,24
3,2010-01-04,24,13.4,24,24,True,24.200000,74.416667,926.600000,1.804167,24,24,24,24
4,2010-01-05,24,50.2,24,24,True,24.641667,73.916667,923.858333,1.933333,24,24,24,24


In [48]:
print("Linhas:", len(df_diario))
print("Data mínima:", df_diario["DATA (YYYY-MM-DD)"].min())
print("Data máxima:", df_diario["DATA (YYYY-MM-DD)"].max())

Linhas: 5844
Data mínima: 2010-01-01 00:00:00
Data máxima: 2025-12-31 00:00:00


In [49]:
colunas_modelagem = [
    "precipitacao_diaria_mm",
    "temperatura_media",
    "umidade_media",
    "pressao_media",
    "vento_medio"
]

resumo_diario = pd.DataFrame({
    "nulos": df_diario[colunas_modelagem].isna().sum(),
    "percentual_nulos": (
        df_diario[colunas_modelagem].isna().mean() * 100
    ).round(2)
})

resumo_diario

,nulos,percentual_nulos
precipitacao_diaria_mm,192,3.29
temperatura_media,46,0.79
umidade_media,39,0.67
pressao_media,39,0.67
vento_medio,39,0.67


## 9. Validação e caracterização da base diária

Após converter os horários da origem UTC para `America/Sao_Paulo`, os registros foram agregados pelo dia civil local. O resultado contém 5.844 datas entre 2010 e 2025, com os acumulados da precipitação indisponíveis nos dias em que a cobertura horária não é integral. As médias auxiliares exigem cobertura temporal integral do dia e proporção mínima de registros válidos.

As tabelas seguintes resumem a disponibilidade no produto diário **corrigido**. Os resultados dos notebooks antigos são referentes à versão UTC e precisam ser recalculados antes de atualizar o relatório do TCC.

In [50]:
resumo_base_final = pd.DataFrame({
    "Característica": [
        "Estação",
        "Período",
        "Frequência original",
        "Registros horários",
        "Frequência final",
        "Dias da série",
        "Variável-alvo",
        "Variáveis auxiliares principais"
    ],
    "Resultado": [
        "A701 - Mirante de Santana",
        "2010-2025",
        "Horária",
        f"{len(df):,}".replace(",", "."),
        "Diária",
        f"{len(df_diario):,}".replace(",", "."),
        "Precipitação diária acumulada",
        "Temperatura, umidade, pressão e vento"
    ]
})

resumo_base_final

,Característica,Resultado
0,Estação,A701 - Mirante de Santana
1,Período,2010-2025
2,Frequência original,Horária
3,Registros horários,140.256
4,Frequência final,Diária
5,Dias da série,5.844
6,Variável-alvo,Precipitação diária acumulada
7,Variáveis auxiliares principais,"Temperatura, umidade, pressão e vento"


In [51]:
resumo_qualidade_final = pd.DataFrame({
    "Variável": [
        "Precipitação diária",
        "Temperatura média",
        "Umidade média",
        "Pressão média",
        "Vento médio"
    ],
    "Dias válidos": [
        df_diario["precipitacao_diaria_mm"].notna().sum(),
        df_diario["temperatura_media"].notna().sum(),
        df_diario["umidade_media"].notna().sum(),
        df_diario["pressao_media"].notna().sum(),
        df_diario["vento_medio"].notna().sum()
    ],
    "Dias indisponíveis": [
        df_diario["precipitacao_diaria_mm"].isna().sum(),
        df_diario["temperatura_media"].isna().sum(),
        df_diario["umidade_media"].isna().sum(),
        df_diario["pressao_media"].isna().sum(),
        df_diario["vento_medio"].isna().sum()
    ]
})

resumo_qualidade_final["Disponibilidade (%)"] = (
    resumo_qualidade_final["Dias válidos"]
    / len(df_diario)
    * 100
).round(2)

resumo_qualidade_final

,Variável,Dias válidos,Dias indisponíveis,Disponibilidade (%)
0,Precipitação diária,5652,192,96.71
1,Temperatura média,5798,46,99.21
2,Umidade média,5805,39,99.33
3,Pressão média,5805,39,99.33
4,Vento médio,5805,39,99.33


## 10. Exportação para a camada Gold — versão local

O novo arquivo possui um nome distinto do anterior para permitir a comparação entre as versões e evitar substituir os dados utilizados nos experimentos já documentados. Somente depois de revisar os resultados e atualizar os notebooks posteriores deverá ser escolhida a versão definitiva.

**Importante:** a agregação abaixo utiliza o calendário civil local de São Paulo. A definição de intervalo do acumulado horário do INMET deverá estar documentada no relatório metodológico.

In [52]:
from pathlib import Path

pasta_gold = Path("../data/gold")
pasta_gold.mkdir(parents=True, exist_ok=True)

# Saída nova: NÃO sobrescreve a base histórica usada na modelagem original.
arquivo_gold = pasta_gold / "a701_2010_2025_diario_horario_sp.parquet"

df_diario.to_parquet(arquivo_gold, index=False)

print(f"Nova base Gold salva em: {arquivo_gold}")
print("Dias:", len(df_diario))
print("Precipitação válida:", df_diario["precipitacao_diaria_mm"].notna().sum())

Nova base Gold salva em: ..\data\gold\a701_2010_2025_diario_horario_sp.parquet
Dias: 5844
Precipitação válida: 5652


## 11. Síntese da etapa revisada

A padronização temporal foi atualizada para interpretar explicitamente os timestamps do INMET como UTC e convertê-los ao fuso histórico `America/Sao_Paulo`. A agregação da precipitação utiliza o dia civil local, com exigência de todas as **23, 24 ou 25** observações horárias esperadas, conforme o calendário. Dias incompletos são mantidos como ausentes, nunca substituídos por zero.

Na auditoria de correção, o período de 2010–2025 apresentou **5.844 dias, 5.652 acumulados válidos e 192 indisponíveis**. A data local parcial de 31/12/2009 foi excluída, assim como o acumulado incompleto em 31/12/2025. Esses resultados devem ser conferidos com a saída gerada pela execução integral do notebook.

As médias diárias de temperatura, umidade, pressão e vento também foram migradas para o calendário local e requerem pelo menos 83,3% de medições válidas, arredondadas para cima, além da cobertura completa dos horários da data. A análise da radiação por hora passou a utilizar hora local.

O resultado foi exportado para um **novo arquivo Gold**, preservando a versão original. Os notebooks de EDA, engenharia de atributos, modelagem, avaliação e consolidação ainda dependem da versão anterior e precisam ser refeitos com a base corrigida. Os resultados numéricos originais não devem ser atribuídos à nova base sem reexecução.

**Ressalva metodológica:** a atribuição da precipitação horária ao intervalo físico de medição deverá ser confirmada na documentação oficial dos arquivos usados. Não foi aplicada qualquer correção arbitrária adicional de uma hora.